In [ ]:
# Run this first. Packages are installed into the active Jupyter kernel.
import shutil
import subprocess
import sys

if sys.version_info < (3, 14):
    raise RuntimeError("qlang requires Python 3.14 or newer. Select a compatible kernel and rerun this cell.")

requirements = [
    "qlang @ git+https://github.com/asherhe/qlang.git",
    "ipywidgets",
    "matplotlib",
]

uv = shutil.which("uv")
pip_command = [sys.executable, "-m", "pip", "install", "--upgrade", *requirements]
print("Installing into the active kernel:", sys.executable)
if uv:
    try:
        subprocess.check_call([uv, "pip", "install", "--python", sys.executable, "--upgrade", *requirements])
    except (OSError, subprocess.CalledProcessError):
        print("uv could not install into this kernel; falling back to pip.")
        subprocess.check_call(pip_command)
else:
    subprocess.check_call(pip_command)


# qlang live schematic

This standalone notebook compiles symbolic predicates into Qiskit phase-oracle circuits. Run the first cell before using the dashboard. It installs packages into the Python environment behind the current Jupyter kernel. To keep them inside this repository's uv environment, launch Jupyter with that environment and choose `.venv\Scripts\python` as the kernel.

> **Interactive viewer required:** the dashboard is a live `ipywidgets` view. If a static notebook preview reports that it cannot find an `ipywidgets` kernel, open this file in JupyterLab or Jupyter Notebook and run the cells there.


## From symbolic predicate to quantum circuit

The dashboard follows qlang's compilation pipeline:

1. **Symbolic parsing** creates `Bit` and unsigned-integer expression nodes from the declared variables and your predicate.
2. **Optimization and bit-blasting** simplify the expression and lower supported arithmetic and Boolean operations to reversible bit-level gadgets.
3. **Qiskit circuit generation** allocates input and workspace qubits, computes the predicate, marks satisfying basis states with a phase flip, and uncomputes the workspace.

The result is a phase oracle, so the expression must evaluate to a Boolean predicate. The expression field accepts declared names, non-negative integer constants, `+`, `-`, `*`, `<<`, `>>`, `&`, `|`, `^`, `~`, and one comparison. Use parentheses around comparisons when combining them. Python `and`, `or`, and `not` are intentionally not accepted.


In [ ]:
import ast
import re

import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import HTML, clear_output, display

import qlang


## Live compiler

Edit any field below. Declarations and domains use one variable per line. Write `x = 3` for a three-bit unsigned integer or `flag = bit` for a Boolean; domains accept an inclusive range such as `x = 0..7` or a value list such as `x = 0, 2, 5`.


In [ ]:
# 16px line height + 16px padding/border allowance = 32px for one line.
expression_input = widgets.Text(
    value="x + y == 5",
    placeholder="(x + y == 5) & (flag | ~flag)",
    continuous_update=True,
    layout=widgets.Layout(width="100%", height="32px", overflow="hidden"),
)
# 3 × 16px line height + 16px padding/border allowance = 64px.
declarations_input = widgets.Textarea(
    value="x = 3\ny = 3",
    placeholder="x = 3\nflag = bit",
    rows=3,
    continuous_update=True,
    layout=widgets.Layout(width="100%", height="64px", overflow="hidden"),
)
domains_input = widgets.Textarea(
    value="x = 0..7\ny = 0..7",
    placeholder="x = 0..7\ny = 0, 2, 5",
    rows=3,
    continuous_update=True,
    layout=widgets.Layout(width="100%", height="64px", overflow="hidden"),
)
for control in (expression_input, declarations_input, domains_input):
    control.add_class("qlang-input")

field_label = lambda text: widgets.HTML(f"<strong>{text}</strong>")
expression_field = widgets.VBox([field_label("Expression"), expression_input], layout=widgets.Layout(width="100%", min_width="0", overflow="hidden"))
declarations_field = widgets.VBox([field_label("Variables"), declarations_input], layout=widgets.Layout(width="49%", min_width="0", overflow="hidden"))
domains_field = widgets.VBox([field_label("Domains"), domains_input], layout=widgets.Layout(width="49%", min_width="0", overflow="hidden"))
configuration_fields = widgets.HBox(
    [declarations_field, domains_field],
    layout=widgets.Layout(width="100%", min_width="0", justify_content="space-between", overflow="hidden"),
)
for container in (expression_field, declarations_field, domains_field, configuration_fields):
    container.add_class("qlang-input-wrapper")

status = widgets.HTML("<em>Waiting for the first compilation.</em>")
circuit_output = widgets.Output(
    layout=widgets.Layout(width="100%", min_width="0", overflow="auto", padding="8px")
)
with circuit_output:
    display(HTML("<em>The compiled circuit will appear here.</em>"))

circuit_panel = widgets.VBox(
    [widgets.HTML("<h4 style='margin: 0 0 8px'>Compiled phase oracle</h4>"), circuit_output],
    layout=widgets.Layout(width="100%", min_width="0", border="1px solid #cbd5e1", padding="12px", overflow="hidden"),
)
circuit_panel.add_class("qlang-circuit-panel")

dashboard_styles = widgets.HTML(
    """<style>
    .qlang-dashboard, .qlang-dashboard .widget-box, .qlang-dashboard .widget-output {
        box-sizing: border-box;
        max-width: 100%;
    }
    .qlang-dashboard .qlang-input-wrapper,
    .qlang-dashboard .qlang-input,
    .qlang-dashboard .qlang-input textarea {
        box-sizing: border-box !important;
        min-width: 0 !important;
        max-width: 100% !important;
        width: 100% !important;
    }
    .qlang-dashboard .qlang-input { overflow-x: hidden !important; }
    .qlang-dashboard textarea.widget-input {
        overflow: hidden !important;
        overflow-wrap: anywhere;
        resize: none;
    }
    .qlang-circuit-panel { transition: opacity 160ms ease; }
    .qlang-circuit-panel.qlang-dimmed { opacity: 0.35; }
    </style>"""
)
dashboard = widgets.VBox(
    [dashboard_styles, expression_field, configuration_fields, status, circuit_panel],
    layout=widgets.Layout(width="90%", max_width="90%", min_width="0", margin="0 auto", grid_gap="10px", overflow="hidden"),
)
dashboard.add_class("qlang-dashboard")
display(dashboard)


In [ ]:
from html import escape
from keyword import iskeyword

_LINE = re.compile(r"^\s*([A-Za-z_]\w*)\s*=\s*(.*?)\s*$")
_RANGE = re.compile(r"^(\d+)\s*\.\.\s*(\d+)$")
_LIST = re.compile(r"^\d+(?:\s*,\s*\d+)*$")
_BINARY_OPERATORS = {ast.Add, ast.Sub, ast.Mult, ast.LShift, ast.RShift, ast.BitAnd, ast.BitOr, ast.BitXor}
_COMPARISON_OPERATORS = {ast.Eq, ast.NotEq, ast.Lt, ast.LtE, ast.Gt, ast.GtE}


def _lines(text, label):
    parsed = {}
    for line_number, raw_line in enumerate(text.splitlines(), start=1):
        if not raw_line.strip():
            continue
        match = _LINE.fullmatch(raw_line)
        if not match:
            raise ValueError(f"{label} line {line_number} must look like 'name = value'")
        name, value = match.groups()
        if iskeyword(name):
            raise ValueError(f"{label} line {line_number} uses Python keyword '{name}'")
        if name in parsed:
            raise ValueError(f"{label} declares '{name}' more than once")
        parsed[name] = value
    if not parsed:
        raise ValueError(f"Provide at least one {label.lower()} entry")
    return parsed


def _variables(text):
    declarations = _lines(text, "Variable")
    variables, widths = {}, {}
    for name, declaration in declarations.items():
        if declaration == "bit":
            variables[name] = qlang.var_bit(name)
            widths[name] = 1
            continue
        if not re.fullmatch(r"[1-9]\d*", declaration):
            raise ValueError(f"Variable '{name}' needs a positive width or the word 'bit'")
        width = int(declaration)
        variables[name] = qlang.uint(name, width=width)
        widths[name] = width
    return variables, widths


def _domains(text, widths):
    declarations = _lines(text, "Domain")
    missing = set(widths) - set(declarations)
    unexpected = set(declarations) - set(widths)
    if missing:
        raise ValueError(f"Missing domains for: {', '.join(sorted(missing))}")
    if unexpected:
        raise ValueError(f"Unknown domain variables: {', '.join(sorted(unexpected))}")

    domains = {}
    for name, declaration in declarations.items():
        range_match = _RANGE.fullmatch(declaration)
        if range_match:
            start, stop = map(int, range_match.groups())
            if start > stop:
                raise ValueError(f"Domain '{name}' has a descending range")
            values = tuple(range(start, stop + 1))
        elif _LIST.fullmatch(declaration):
            values = tuple(sorted({int(value.strip()) for value in declaration.split(",")}))
        else:
            raise ValueError(f"Domain '{name}' needs 'start..stop' or comma-separated integers")
        limit = 1 << widths[name]
        if not values or values[0] < 0 or values[-1] >= limit:
            raise ValueError(f"Domain '{name}' must fit its {widths[name]}-bit declaration (0..{limit - 1})")
        domains[name] = values
    return domains


def _validate_expression(tree, names):
    def visit(node):
        if isinstance(node, ast.Expression):
            visit(node.body)
        elif isinstance(node, ast.Name):
            if node.id not in names:
                raise ValueError(f"Unknown variable '{node.id}'")
        elif isinstance(node, ast.Constant):
            if type(node.value) is not int or node.value < 0:
                raise ValueError("Only non-negative integer constants are allowed")
        elif isinstance(node, ast.BinOp):
            if type(node.op) not in _BINARY_OPERATORS:
                raise ValueError("That binary operator is not supported")
            visit(node.left)
            visit(node.right)
        elif isinstance(node, ast.UnaryOp):
            if type(node.op) is not ast.Invert:
                raise ValueError("Only symbolic '~' is allowed as a unary operator")
            visit(node.operand)
        elif isinstance(node, ast.Compare):
            if len(node.ops) != 1 or type(node.ops[0]) not in _COMPARISON_OPERATORS:
                raise ValueError("Use exactly one supported comparison per expression")
            visit(node.left)
            visit(node.comparators[0])
        else:
            raise ValueError(f"Unsupported expression syntax: {type(node).__name__}")

    visit(tree)


def _predicate(text, variables):
    tree = ast.parse(text, mode="eval")
    _validate_expression(tree, variables)
    predicate = eval(compile(tree, "<qlang expression>", "eval"), {"__builtins__": {}}, variables)
    if not isinstance(predicate, qlang.Bit):
        raise TypeError("The expression must produce a Boolean predicate, such as 'x + y == 5'")
    return predicate


def _set_error(message):
    expression_input.layout.border = "2px solid #dc2626"
    circuit_panel.add_class("qlang-dimmed")
    status.value = f"<span style='color: #b91c1c'><strong>Compilation error:</strong> {escape(str(message))}</span>"
    with circuit_output:
        clear_output(wait=True)
        display(HTML("<em>Fix the inputs to render a circuit.</em>"))


def _set_success(oracle, domains):
    expression_input.layout.border = ""
    circuit_panel.remove_class("qlang-dimmed")
    domain_summary = "; ".join(f"{name}: {list(values)}" for name, values in domains.items())
    status.value = "<span style='color: #166534'><strong>Compiled successfully.</strong> Domains validated; no search was run.</span>"
    with circuit_output:
        clear_output(wait=True)
        oracle_info = (
            f"inputs: {oracle.inputs}\n"
            f"qubits: {oracle.circuit.num_qubits}\n"
            f"ancillas: {oracle.total_ancillas}\n"
            f"gates: {oracle.circuit.size()}\n"
            f"validated domains: {domain_summary}"
        )
        display(HTML(
            "<pre style='font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace; "
            "margin: 0 0 8px; white-space: pre-wrap'>"
            f"{escape(oracle_info)}</pre>"
        ))
        figure = oracle.circuit.draw("mpl", fold=-1)
        display(figure)
        plt.close(figure)


_refreshing = False


def refresh(_change=None):
    global _refreshing
    if _refreshing:
        return
    _refreshing = True
    try:
        qlang.ASTNode.reset()
        variables, widths = _variables(declarations_input.value)
        domains = _domains(domains_input.value, widths)
        predicate = _predicate(expression_input.value, variables)
        _set_success(qlang.compile_oracle(predicate), domains)
    except Exception as error:
        _set_error(error)
    finally:
        _refreshing = False


for control in (expression_input, declarations_input, domains_input):
    control.observe(refresh, names="value")

refresh()


## Where this circuit fits

A compiled phase oracle can be used by **Grover search** to amplify assignments that satisfy the predicate; the validated domains describe the values you would pass to `qlang.search` when you choose to run it. For optimization, **QAOA** uses the same symbolic constraint together with an optional symbolic unsigned objective to form a cost operator. This dashboard deliberately stops at compilation so edits stay fast and errors remain easy to inspect.

The circuit includes temporary workspace qubits. qlang reverses the temporary computation after applying the phase mark, returning that workspace to zero so the oracle is suitable for larger quantum algorithms.
